### Preprocessing

In [1]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler

# ======================================
# 1️⃣ Load dataset with 3-row header
# ======================================
path = r"E:\ML_Project\polarH10_2\all_participants_features_polarh10.csv"

# Read header rows first (3 rows)
header_rows = pd.read_csv(path, nrows=3, header=None)

# Build flattened column names
cols = []
for i in range(header_rows.shape[1]):
    top = str(header_rows.iat[0, i]) if pd.notna(header_rows.iat[0, i]) else ""
    mid = str(header_rows.iat[1, i]) if pd.notna(header_rows.iat[1, i]) else ""
    bot = str(header_rows.iat[2, i]) if pd.notna(header_rows.iat[2, i]) else ""

    if i == 0 or bot.lower() == "datetime":
        cols.append("datetime")
    elif top and mid:
        cols.append(f"{top}_{mid}")
    elif top:
        cols.append(top)
    elif mid:
        cols.append(mid)
    else:
        cols.append(f"col_{i}")

# Read full data skipping first 3 rows
df = pd.read_csv(path, skiprows=3, names=cols, low_memory=False)

# ======================================
# 2️⃣ Datetime + Participant handling
# ======================================
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = df.dropna(subset=['datetime'])

if 'participant' not in df.columns:
    raise ValueError("❌ 'participant' column not found in dataset!")

df = df.sort_values(by=['participant', 'datetime']).reset_index(drop=True)

print("✅ Data loaded and sorted by participant+datetime")
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

# ======================================
# 3️⃣ Signal Processing
# ======================================
def butter_bandpass_filter(data, lowcut, highcut, fs=1, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = min(highcut / nyq, 0.99)
    if low >= high:
        raise ValueError("Invalid frequency band — skipping filter.")
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

# --- ECG filtering (analogous to BVP filtering) ---
if 'ecg_mean' in df.columns:
    fs = 1
    if fs <= 2:
        print("ℹ️ Skipping ECG filtering — 1 Hz data insufficient for 5–15 Hz band.")
        df['ecg_filtered'] = df['ecg_mean']
    else:
        df['ecg_filtered'] = np.nan
        for pid, group in df.groupby('participant'):
            try:
                filtered = butter_bandpass_filter(group['ecg_mean'].fillna(0).values, 5, 15, fs=fs)
                df.loc[group.index, 'ecg_filtered'] = filtered
            except Exception as e:
                print(f"⚠️ Skipping ECG filter for {pid}: {e}")

# --- ACC magnitude + smoothing ---
if all(col in df.columns for col in ['x_mean', 'y_mean', 'z_mean']):
    df['acc_mag'] = np.sqrt(df['x_mean']**2 + df['y_mean']**2 + df['z_mean']**2)
    df['acc_mag_smooth'] = df.groupby('participant')['acc_mag'].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )
    print("✅ ACC magnitude computed")

    # Drop raw accelerometer columns
    df = df.drop(columns=['x_mean', 'y_mean', 'z_mean', 'x_std', 'y_std', 'z_std'], errors='ignore')
    print("✅ Dropped x/y/z columns")

# --- Remove temperature columns (not present in Polar) ---
df = df.drop(columns=[col for col in df.columns if 'temp' in col.lower()], errors='ignore')
print("✅ Removed temperature-related columns (if any)")

# ======================================
# 4️⃣ Per-participant Normalization
# ======================================
normalized = []
for pid, group in df.groupby('participant'):
    num_cols = group.select_dtypes(include=[np.number]).columns
    num_cols = [c for c in num_cols if group[c].std(skipna=True) > 0]
    group[num_cols] = StandardScaler().fit_transform(group[num_cols])
    normalized.append(group)

df = pd.concat(normalized).reset_index(drop=True)

print("✅ Normalized per participant")
print("✅ Final shape:", df.shape)
print(df.head())

# ======================================
# 5️⃣ Save Preprocessed Data
# ======================================
output_path = r"E:\ML_Project\polarH10_2\processed_polar.csv"
df.to_csv(output_path, index=False)

print(f"✅ Preprocessing complete. Saved as {output_path}")


✅ Data loaded and sorted by participant+datetime
Columns: ['datetime', 'x_mean', 'x_std', 'y_mean', 'y_std', 'z_mean', 'z_std', 'ecg_mean', 'ecg_std', 'hr_mean', 'hr_std', 'hrv_mean', 'hrv_std', 'ibi_mean', 'ibi_std', 'participant']
Shape: (94848, 16)
ℹ️ Skipping ECG filtering — 1 Hz data insufficient for 5–15 Hz band.
✅ ACC magnitude computed
✅ Dropped x/y/z columns
✅ Removed temperature-related columns (if any)
✅ Normalized per participant
✅ Final shape: (94848, 13)
             datetime   ecg_mean    ecg_std   hr_mean    hr_std  hrv_mean  \
0 2024-03-18 13:15:29  29.871600        NaN  0.115432       NaN -0.519081   
1 2024-03-18 13:15:30  29.871600  -0.203567  0.094339 -0.877627 -0.519081   
2 2024-03-18 13:15:31  24.918635   9.038398  0.059183 -0.543549 -0.519081   
3 2024-03-18 13:15:32  19.297100  14.066969  0.041605 -0.554757 -0.549075   
4 2024-03-18 13:15:33  15.489459  15.186940  0.022621 -0.496459 -0.595067   

    hrv_std  ibi_mean   ibi_std participant  ecg_filtered   acc_

In [2]:
# Get descriptive statistics for 'hrv_mean', 'hrv_std', and 'ibi_mean'
df[['hrv_mean', 'hrv_std', 'ibi_mean']].describe()


,hrv_mean,hrv_std,ibi_mean
count,9.484800e+04,9.482400e+04,9.484800e+04
mean,1.318483e-17,-9.591398e-18,-7.910901e-17
std,1.000005e+00,1.000005e+00,1.000005e+00
min,-2.138364e+00,-1.596190e+00,-4.365250e+00
25%,-5.228341e-01,-5.311256e-01,-3.159170e-01
50%,-1.697708e-01,-2.477902e-01,2.289832e-01
75%,3.937441e-01,2.256698e-01,5.935350e-01
max,1.055621e+01,1.047923e+01,2.541993e+01


In [3]:
import pandas as pd
import numpy as np

# --- Step 1: Load preprocessed Polar H10 data ---
df = pd.read_csv(r"E:\ML_Project\polarH10_2\processed_polar.csv")

# --- Step 2: Define stress thresholds ---
# Adjust thresholds based on z-score normalized values
def assign_stress_label(row):
    # High stress: low HRV_mean OR low HRV_std OR low IBI_mean
    if (row['hrv_mean'] < -0.5) or (row['hrv_std'] < -0.5) or (row['ibi_mean'] < -0.5):
        return 'High'
    # Low stress: all values high
    elif (row['hrv_mean'] > 0.5) and (row['hrv_std'] > 0.5) and (row['ibi_mean'] > 0.5):
        return 'Low'
    # Otherwise Medium
    else:
        return 'Medium'

# --- Step 3: Apply labeling ---
df['stress_level'] = df.apply(assign_stress_label, axis=1)

# --- Step 4: Check class distribution ---
print(df['stress_level'].value_counts())

# --- Step 5: Drop HRV/IBI features to prevent leakage for model training ---
model_features = df.drop(columns=['hrv_mean', 'hrv_std', 'ibi_mean', 'ibi_std', 'stress_level'])

# --- Step 6: Save labeled dataset ---
output_path = r"E:\ML_Project\polarH10_2\polarH10_stress_labeled.csv"
df.to_csv(output_path, index=False)
print(f"✅ Stress labels created and saved: {output_path}")


stress_level
Medium    48303
High      40469
Low        6076
Name: count, dtype: int64
✅ Stress labels created and saved: E:\ML_Project\polarH10_2\polarH10_stress_labeled.csv


In [5]:
import pandas as pd
import os

# Load preprocessed Polar H10 data with stress labels
df = pd.read_csv(r"E:\ML_Project\polarH10_2\polarH10_stress_labeled.csv")

# Define participant groups
train_participants = ['p_03', 'p_04', 'p_05', 'p_06', 'p_07', 'p_08',
                      'p_11', 'p_13', 'p_15', 'p_16', 'p_18', 'p_20', 'p_21', 'p_23', 'p_24']
val_participants   = ['p_12', 'p_14', 'p_19']
test_participants  = ['p_01', 'p_02', 'p_09', 'p_10', 'p_17', 'p_22']

# Create splits
train_df = df[df['participant'].isin(train_participants)].copy()
val_df   = df[df['participant'].isin(val_participants)].copy()
test_df  = df[df['participant'].isin(test_participants)].copy()

# For test set, drop labels to simulate unseen data
test_df = test_df.drop(columns=['stress_level'], errors='ignore')

# Check split sizes
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

save_dir = r"E:\ML_Project\polarH10_2"

# Save CSVs
train_path = os.path.join(save_dir, "train_split.csv")
val_path   = os.path.join(save_dir, "val_split.csv")
test_path  = os.path.join(save_dir, "test_split.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("✅ Splits saved successfully!")
print(f"Train: {train_path}")
print(f"Validation: {val_path}")
print(f"Test: {test_path}")



Train shape: (59521, 14)
Validation shape: (11863, 14)
Test shape: (23464, 13)
✅ Splits saved successfully!
Train: E:\ML_Project\polarH10_2\train_split.csv
Validation: E:\ML_Project\polarH10_2\val_split.csv
Test: E:\ML_Project\polarH10_2\test_split.csv


### Start from here 

In [21]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# ---------------------------
# Load Splits
# ---------------------------
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

# ---------------------------
# Features & Target
# ---------------------------
features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std', 
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val   = val_df[features]
y_val   = val_df[target]

# ✅ Test has features only (no labels)
X_test  = test_df[features]

# ---------------------------
# Scaling
# ---------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# ---------------------------
# Model Definition
# ---------------------------
rf = RandomForestClassifier(
    n_estimators=2000,
    max_depth=6,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# ---------------------------
# Train Model
# ---------------------------
rf.fit(X_train_scaled, y_train)

# ---------------------------
# Validation Evaluation
# ---------------------------
y_val_pred = rf.predict(X_val_scaled)

print("\n📊 Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# ---------------------------
# ✅ Test Predictions (NO accuracy possible)
# ---------------------------
test_predictions = rf.predict(X_test_scaled)

test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_predictions

test_output.to_csv("test_predictions_H10_RF.csv", index=False)
print("\n✅ Test predictions generated")


📊 Validation Results
              precision    recall  f1-score   support

        High       0.83      0.73      0.78      4491
         Low       0.31      0.65      0.42       694
      Medium       0.83      0.80      0.81      6678

    accuracy                           0.77     11863
   macro avg       0.66      0.73      0.67     11863
weighted avg       0.80      0.77      0.78     11863

Confusion Matrix:
 [[3285  294  912]
 [  46  450  198]
 [ 629  694 5355]]
Accuracy: 0.7662
Weighted F1 Score: 0.7777

✅ Test predictions generated


### LogisticRegression

In [20]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# ---------------------------
# Load Splits
# ---------------------------
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

# ---------------------------
# Features & Target
# ---------------------------
features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std', 
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val   = val_df[features]
y_val   = val_df[target]

X_test  = test_df[features]

# ---------------------------
# ✅ Impute Missing Values
# ---------------------------
imputer = SimpleImputer(strategy="mean")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

# ---------------------------
# ✅ Scaling
# ---------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# ---------------------------
# ✅ Logistic Regression
# ---------------------------
model = LogisticRegression(
    multi_class="multinomial",
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs"
)

model.fit(X_train_scaled, y_train)

# ---------------------------
# Validation Evaluation
# ---------------------------
y_val_pred = model.predict(X_val_scaled)

print("\n📊 Logistic Regression - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# ---------------------------
# Test Predictions
# ---------------------------
test_predictions = model.predict(X_test_scaled)

test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_predictions
test_output.to_csv("test_predictions_H10_LogReg.csv", index=False)

print("\n✅ Test predictions generated (with imputation)")

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



📊 Logistic Regression - Validation Results
              precision    recall  f1-score   support

        High       0.70      0.76      0.73      4491
         Low       0.31      0.87      0.45       694
      Medium       0.82      0.62      0.71      6678

    accuracy                           0.69     11863
   macro avg       0.61      0.75      0.63     11863
weighted avg       0.75      0.69      0.70     11863

Confusion Matrix:
 [[3392  245  854]
 [  28  602   64]
 [1401 1123 4154]]
Accuracy: 0.6868
Weighted F1 Score: 0.7003

✅ Test predictions generated (with imputation)


### DecisionTreeClassifier

In [19]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Load
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# Impute
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

# Model
model = DecisionTreeClassifier(
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_imp, y_train)

# Validation
y_val_pred = model.predict(X_val_imp)

print("\n📊 Decision Tree - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# Test
test_pred = model.predict(X_test_imp)
test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred
test_output.to_csv("test_predictions_H10_DecisionTree.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_DecisionTree.csv")


📊 Decision Tree - Validation Results
              precision    recall  f1-score   support

        High       0.85      0.61      0.71      4491
         Low       0.33      0.76      0.46       694
      Medium       0.76      0.80      0.78      6678

    accuracy                           0.73     11863
   macro avg       0.65      0.72      0.65     11863
weighted avg       0.77      0.73      0.74     11863

Confusion Matrix:
 [[2723  261 1507]
 [  12  528  154]
 [ 484  822 5372]]
Accuracy: 0.7269
Weighted F1 Score: 0.7353

✅ Test predictions saved: test_predictions_H10_DecisionTree.csv


### XGBClassifier

In [18]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# ---------------------------
# Load Data
# ---------------------------
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

# ---------------------------
# Features & Target
# ---------------------------
features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# ---------------------------
# ✅ Encode String Labels → Numbers
# ---------------------------
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)   # ['High','Low','Medium'] → [0,1,2]
y_val_enc   = le.transform(y_val)

# ---------------------------
# Model
# ---------------------------
model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.005,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train_enc)

# ---------------------------
# Validation Predictions
# ---------------------------
y_val_pred_enc = model.predict(X_val)
y_val_pred = le.inverse_transform(y_val_pred_enc)

print("\n📊 XGBoost - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# ---------------------------
# Test Predictions
# ---------------------------
test_pred_enc = model.predict(X_test)
test_pred = le.inverse_transform(test_pred_enc)

test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred

test_output.to_csv("test_predictions_H10_XGBoost.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_XGBoost.csv")


📊 XGBoost - Validation Results
              precision    recall  f1-score   support

        High       0.82      0.63      0.71      4491
         Low       0.56      0.14      0.23       694
      Medium       0.73      0.90      0.81      6678

    accuracy                           0.75     11863
   macro avg       0.70      0.56      0.58     11863
weighted avg       0.75      0.75      0.74     11863

Confusion Matrix:
 [[2847    2 1642]
 [  38   99  557]
 [ 596   75 6007]]
Accuracy: 0.7547
Weighted F1 Score: 0.7381

✅ Test predictions saved: test_predictions_H10_XGBoost.csv


### CatBoostClassifier

In [22]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Load
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# Model
model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.005,
    loss_function="MultiClass",
    random_seed=42,
    verbose=0
)

model.fit(X_train, y_train)

# Validation
y_val_pred = model.predict(X_val)

print("\n📊 CatBoost - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# ---------------------------
# ✅ Test Predictions
# ---------------------------
test_pred = model.predict(X_test)

# Flatten if needed (CatBoost returns 2D)
if test_pred.ndim > 1:
    test_pred = test_pred.ravel()

test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred

test_output.to_csv("test_predictions_H10_CatBoost.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_CatBoost.csv")


📊 CatBoost - Validation Results
              precision    recall  f1-score   support

        High       0.85      0.67      0.75      4491
         Low       0.63      0.25      0.36       694
      Medium       0.76      0.91      0.83      6678

    accuracy                           0.78     11863
   macro avg       0.75      0.61      0.64     11863
weighted avg       0.78      0.78      0.77     11863

Confusion Matrix:
 [[2998    2 1491]
 [  42  172  480]
 [ 497   97 6084]]
Accuracy: 0.7801
Weighted F1 Score: 0.7685

✅ Test predictions saved: test_predictions_H10_CatBoost.csv


### LightGBM

In [23]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Load
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# Model
model = lgb.LGBMClassifier(
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.005,
    class_weight="balanced",
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)

# Validation
y_val_pred = model.predict(X_val)

print("\n📊 LightGBM - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# Test
test_pred = model.predict(X_test)
test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred
test_output.to_csv("test_predictions_H10_LightGBM.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_LightGBM.csv")


📊 LightGBM - Validation Results
              precision    recall  f1-score   support

        High       0.79      0.63      0.70      4491
         Low       0.23      0.51      0.31       694
      Medium       0.74      0.75      0.74      6678

    accuracy                           0.69     11863
   macro avg       0.59      0.63      0.59     11863
weighted avg       0.73      0.69      0.70     11863

Confusion Matrix:
 [[2837  221 1433]
 [  39  354  301]
 [ 705  982 4991]]
Accuracy: 0.6897
Weighted F1 Score: 0.7038

✅ Test predictions saved: test_predictions_H10_LightGBM.csv


### MLP

In [24]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Load
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# Imputation
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# Model
model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    max_iter=2000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# Validation
y_val_pred = model.predict(X_val_scaled)

print("\n📊 MLP Neural Network - Validation Results")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, y_val_pred, average='weighted'):.4f}")

# Test
test_pred = model.predict(X_test_scaled)
test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred
test_output.to_csv("test_predictions_H10_MLP.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_MLP.csv")


📊 MLP Neural Network - Validation Results
              precision    recall  f1-score   support

        High       0.65      0.59      0.62      4491
         Low       0.33      0.29      0.31       694
      Medium       0.71      0.75      0.73      6678

    accuracy                           0.66     11863
   macro avg       0.56      0.55      0.55     11863
weighted avg       0.66      0.66      0.66     11863

Confusion Matrix:
 [[2665  102 1724]
 [ 116  203  375]
 [1340  319 5019]]
Accuracy: 0.6648
Weighted F1 Score: 0.6619

✅ Test predictions saved: test_predictions_H10_MLP.csv


### Deep MLP

In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# ---------------------------
# Load Data
# ---------------------------
train_df = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\train_split.csv")
val_df   = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\val_split.csv")
test_df  = pd.read_csv(r"C:\Users\HP\Desktop\Projects\ML Project\H10\test_split.csv")

features = ['ecg_mean', 'ecg_std', 'hr_mean', 'hr_std',
            'ibi_std', 'ecg_filtered', 'acc_mag', 'acc_mag_smooth']
target = 'stress_level'

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]

# ---------------------------
# Encode labels
# ---------------------------
label_map = {"Low": 0, "Medium": 1, "High": 2}
inv_map = {0:"Low", 1:"Medium", 2:"High"}

y_train_enc = train_df[target].map(label_map).values
y_val_enc   = val_df[target].map(label_map).values

y_train_cat = to_categorical(y_train_enc, 3)
y_val_cat   = to_categorical(y_val_enc, 3)

# ---------------------------
# Imputation
# ---------------------------
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

# ---------------------------
# Scaling
# ---------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# ---------------------------
# Deep MLP Model
# ---------------------------
model = Sequential([
    Dense(256, activation='relu', input_shape=(8,)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# Train Model
# ---------------------------
model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=25,
    batch_size=32,
    verbose=1
)

# ---------------------------
# Validation Evaluation
# ---------------------------
val_pred_prob = model.predict(X_val_scaled)
val_pred = np.argmax(val_pred_prob, axis=1)
val_pred_labels = [inv_map[i] for i in val_pred]

print("\n📊 Deep MLP - Validation Results")
print(classification_report(y_val, val_pred_labels))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_labels))
print(f"Accuracy: {accuracy_score(y_val, val_pred_labels):.4f}")
print(f"Weighted F1 Score: {f1_score(y_val, val_pred_labels, average='weighted'):.4f}")

# ---------------------------
# Test Predictions
# ---------------------------
test_pred_prob = model.predict(X_test_scaled)
test_pred = np.argmax(test_pred_prob, axis=1)
test_pred_labels = [inv_map[i] for i in test_pred]

test_output = test_df.copy()
test_output["Predicted_Stress_Level"] = test_pred_labels
test_output.to_csv("test_predictions_H10_DeepMLP.csv", index=False)

print("\n✅ Test predictions saved: test_predictions_H10_DeepMLP.csv")

Epoch 1/25


C:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1861/1861 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7100 - loss: 0.6589 - val_accuracy: 0.7834 - val_loss: 0.6216
Epoch 2/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7426 - loss: 0.5788 - val_accuracy: 0.7748 - val_loss: 0.6556
Epoch 3/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7438 - loss: 0.5681 - val_accuracy: 0.7701 - val_loss: 0.6324
Epoch 4/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7468 - loss: 0.5602 - val_accuracy: 0.7668 - val_loss: 0.6212
Epoch 5/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7517 - loss: 0.5473 - val_accuracy: 0.7725 - val_loss: 0.6476
Epoch 6/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7546 - loss: 0.5406 - val_accuracy: 0.7563 - val_loss: 0.6515
Epoch 7/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7567 - loss: 0.5356 - val_accuracy: 0.7485 - val_loss: 0.7546
Epoch 8/25
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.7567 - loss: 0.5313 - val_accurac